# Download the complete KneeXrayData dataset to Google Drive

This notebook downloads the complete Mendeley Data release used by the Knee Osteoarthritis Severity Grading Dataset. It preserves the original archive, verifies its published SHA-256, safely extracts every file, and validates the expected directory counts.

Use a normal Colab CPU runtime and run every cell in order. Rerunning the notebook resumes an interrupted download and skips files that were already extracted atomically. The Drive destination is `MyDrive/Datasets/KneeXrayData_Mendeley_v1`.


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import hashlib
import json
import os
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

import requests
from tqdm.auto import tqdm

DATASET_PAGE = 'https://data.mendeley.com/datasets/56rmx5bjcr/1'
METADATA_URL = 'https://data.mendeley.com/public-api/datasets/56rmx5bjcr'
MENDELEY_DOWNLOAD_URL = (
    'https://data.mendeley.com/public-files/datasets/56rmx5bjcr/files/'
    '205a9a5d-5de5-4f06-8f63-e430ed23ca44/file_downloaded'
)
STORAGE_DOWNLOAD_URL = (
    'https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/'
    '289dd733-3e9c-491e-b1e7-6b6cb3a58ba5'
)
REQUEST_HEADERS = {
    # Cloudflare may reject the default python-requests user agent in Colab.
    'User-Agent': (
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/126.0 Safari/537.36'
    ),
    'Accept': 'application/json, text/plain, */*',
    'Accept-Encoding': 'identity',
    'Referer': DATASET_PAGE,
}
EXPECTED_ARCHIVE_SIZE = 7_171_571_746
EXPECTED_UNCOMPRESSED_SIZE = 7_708_018_803
EXPECTED_ARCHIVE_ENTRIES = 23_832
EXPECTED_SHA256 = '3b4b93b3a3e2e2f059684f5d2b8bbd05aa7776d0199365fc6db51b93dc667505'

DRIVE_ROOT = Path('/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1')
ARCHIVE_PATH = DRIVE_ROOT / 'KneeXrayData.zip'
EXTRACT_ROOT = DRIVE_ROOT / 'extracted'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Drive destination: {DRIVE_ROOT}')
print('GPU is not required for this notebook.')


Mounted at /content/drive
Drive destination: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1
GPU is not required for this notebook.


In [2]:
http_session = requests.Session()
http_session.headers.update(REQUEST_HEADERS)
# Colab IP addresses can receive HTTP 403 from Mendeley's Cloudflare
# endpoint. This is the public S3 object behind Mendeley's own redirect.
download_url = STORAGE_DOWNLOAD_URL

# Metadata is useful for provenance validation, but it is not required to
# download the verified public storage object. Mendeley's Cloudflare layer
# can return 403 specifically to Google Colab outbound IP addresses.
try:
    response = http_session.get(METADATA_URL, timeout=(30, 60))
    response.raise_for_status()
    metadata = response.json()

    if metadata.get('doi', {}).get('id') != '10.17632/56rmx5bjcr.1':
        raise RuntimeError('Unexpected Mendeley dataset DOI')
    files = metadata.get('files', [])
    if len(files) != 1 or files[0].get('filename') != 'KneeXrayData.zip':
        raise RuntimeError(f'Unexpected Mendeley file inventory: {files}')
    file_metadata = files[0]['content_details']
    if int(file_metadata['size']) != EXPECTED_ARCHIVE_SIZE:
        raise RuntimeError('Published archive size changed')
    if file_metadata['sha256_hash'].lower() != EXPECTED_SHA256:
        raise RuntimeError('Published archive SHA-256 changed')
    print('Official Mendeley metadata check passed.')
except requests.RequestException as error:
    print(f'Metadata request unavailable ({error}).')
    print('Continuing with the verified public Mendeley storage object.')

print(f'Download source: {download_url}')

free_bytes = shutil.disk_usage('/content/drive/MyDrive').free
existing_archive_bytes = ARCHIVE_PATH.stat().st_size if ARCHIVE_PATH.exists() else 0
existing_extracted_bytes = sum(
    path.stat().st_size for path in EXTRACT_ROOT.rglob('*') if path.is_file()
)
remaining_archive_bytes = max(0, EXPECTED_ARCHIVE_SIZE - existing_archive_bytes)
remaining_extracted_bytes = max(0, EXPECTED_UNCOMPRESSED_SIZE - existing_extracted_bytes)
required_bytes = remaining_archive_bytes + remaining_extracted_bytes + 2 * 1024**3
print(f'Google Drive free space: {free_bytes / 1024**3:.2f} GiB')
print(f'Recommended free space: {required_bytes / 1024**3:.2f} GiB')
if free_bytes < required_bytes:
    raise RuntimeError(
        'Google Drive does not have enough free space for the archive, '
        'complete extraction, and a 2 GiB safety margin.'
    )
print('Google Drive capacity check passed.')


Metadata request unavailable (403 Client Error: Forbidden for url: https://data.mendeley.com/public-api/datasets/56rmx5bjcr).
Continuing with the verified public Mendeley storage object.
Download source: https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/289dd733-3e9c-491e-b1e7-6b6cb3a58ba5
Google Drive free space: 62.61 GiB
Recommended free space: 8.88 GiB
Google Drive capacity check passed.


In [3]:
def download_with_resume(url: str, destination: Path, expected_size: int) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, 9):
        existing_size = destination.stat().st_size if destination.exists() else 0
        if existing_size == expected_size:
            print(f'Download already complete: {destination}')
            return
        if existing_size > expected_size:
            raise RuntimeError(
                f'Existing archive is too large: {existing_size} > {expected_size}'
            )

        headers = {'Range': f'bytes={existing_size}-'} if existing_size else {}
        mode = 'ab' if existing_size else 'wb'
        try:
            with http_session.get(
                url,
                headers=headers,
                stream=True,
                allow_redirects=True,
                timeout=(30, 120),
            ) as download:
                download.raise_for_status()
                if existing_size and download.status_code != 206:
                    raise RuntimeError(
                        'The server ignored the resume Range request; refusing to corrupt the archive.'
                    )
                with destination.open(mode) as output, tqdm(
                    total=expected_size,
                    initial=existing_size,
                    unit='B',
                    unit_scale=True,
                    desc='KneeXrayData.zip',
                ) as progress:
                    for chunk in download.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            output.write(chunk)
                            progress.update(len(chunk))
        except (requests.RequestException, OSError) as error:
            if attempt == 8:
                raise
            wait_seconds = min(30, 2**attempt)
            print(f'Download interrupted: {error}')
            print(f'Retrying from the saved byte position in {wait_seconds}s...')
            time.sleep(wait_seconds)

    final_size = destination.stat().st_size
    if final_size != expected_size:
        raise RuntimeError(f'Incomplete download: {final_size} of {expected_size} bytes')


download_with_resume(download_url, ARCHIVE_PATH, EXPECTED_ARCHIVE_SIZE)
print(f'Archive saved to: {ARCHIVE_PATH}')


Download already complete: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/KneeXrayData.zip
Archive saved to: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/KneeXrayData.zip


In [4]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle, tqdm(
        total=path.stat().st_size,
        unit='B',
        unit_scale=True,
        desc='Verifying SHA-256',
    ) as progress:
        for block in iter(lambda: handle.read(16 * 1024 * 1024), b''):
            digest.update(block)
            progress.update(len(block))
    return digest.hexdigest()


actual_sha256 = sha256_file(ARCHIVE_PATH)
print(f'Archive SHA-256: {actual_sha256}')
if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        'Archive checksum mismatch. Do not extract or use this file. '
        'Rename/remove the damaged archive and rerun the download cell.'
    )
print('Archive checksum verified successfully.')


Verifying SHA-256:   0%|          | 0.00/7.17G [00:00<?, ?B/s]

Archive SHA-256: 3b4b93b3a3e2e2f059684f5d2b8bbd05aa7776d0199365fc6db51b93dc667505
Archive checksum verified successfully.


In [5]:
def safe_extract_resumable(archive_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(archive_path, 'r') as archive:
        members = archive.infolist()
        if len(members) != EXPECTED_ARCHIVE_ENTRIES:
            raise RuntimeError(
                f'Unexpected ZIP entry count: {len(members)} != {EXPECTED_ARCHIVE_ENTRIES}'
            )
        for member in tqdm(members, desc='Extracting', unit='entry'):
            target = (destination / member.filename).resolve()
            if target != destination_resolved and destination_resolved not in target.parents:
                raise RuntimeError(f'Unsafe ZIP path: {member.filename}')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists() and target.stat().st_size == member.file_size:
                continue
            partial = target.with_name(target.name + '.partial')
            with archive.open(member, 'r') as source, partial.open('wb') as output:
                shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
            if partial.stat().st_size != member.file_size:
                raise RuntimeError(f'Extracted size mismatch: {member.filename}')
            os.replace(partial, target)


safe_extract_resumable(ARCHIVE_PATH, EXTRACT_ROOT)
print(f'Complete dataset extracted to: {EXTRACT_ROOT}')


Extracting:   0%|          | 0/23832 [00:00<?, ?entry/s]

Complete dataset extracted to: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted


In [6]:
DATA_ROOT = EXTRACT_ROOT / 'KneeXrayData'
expected_folder_counts = {
    'ClsKLData/kneeKL224/train': 5_778,
    'ClsKLData/kneeKL224/val': 826,
    'ClsKLData/kneeKL224/test': 1_656,
    'ClsKLData/kneeKL224/auto_test': 1_526,
    'ClsKLData/kneeKL299/train': 5_778,
    'ClsKLData/kneeKL299/val': 826,
    'ClsKLData/kneeKL299/test': 1_656,
    'ClsKLData/kneeKL299/auto_test': 1_526,
    'DetKneeData/H5/trainH5': 2_889,
    'DetKneeData/H5/valH5': 413,
    'DetKneeData/H5/testH5': 828,
}

validation = {}
for relative, expected_count in expected_folder_counts.items():
    folder = DATA_ROOT / relative
    actual_count = sum(1 for path in folder.rglob('*') if path.is_file())
    validation[relative] = {'expected': expected_count, 'actual': actual_count}
    if actual_count != expected_count:
        raise RuntimeError(
            f'File count mismatch for {relative}: {actual_count} != {expected_count}'
        )

extension_counts = Counter(
    path.suffix.lower() for path in DATA_ROOT.rglob('*') if path.is_file()
)
expected_extensions = {'.png': 19_572, '.h5': 4_130, '.pth': 28}
for extension, expected_count in expected_extensions.items():
    if extension_counts[extension] != expected_count:
        raise RuntimeError(
            f'{extension} count mismatch: {extension_counts[extension]} != {expected_count}'
        )

validation_summary = {
    'dataset_page': DATASET_PAGE,
    'doi': '10.17632/56rmx5bjcr.1',
    'license': 'CC BY 4.0',
    'archive': str(ARCHIVE_PATH),
    'archive_size': ARCHIVE_PATH.stat().st_size,
    'archive_sha256': actual_sha256,
    'extracted_root': str(DATA_ROOT),
    'folder_counts': validation,
    'extension_counts': dict(extension_counts),
}
summary_path = DRIVE_ROOT / 'validation_summary.json'
summary_path.write_text(json.dumps(validation_summary, indent=2), encoding='utf-8')
print(json.dumps(validation_summary, indent=2))
print(f'Validation summary saved to: {summary_path}')


{
  "dataset_page": "https://data.mendeley.com/datasets/56rmx5bjcr/1",
  "doi": "10.17632/56rmx5bjcr.1",
  "license": "CC BY 4.0",
  "archive": "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/KneeXrayData.zip",
  "archive_size": 7171571746,
  "archive_sha256": "3b4b93b3a3e2e2f059684f5d2b8bbd05aa7776d0199365fc6db51b93dc667505",
  "extracted_root": "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData",
  "folder_counts": {
    "ClsKLData/kneeKL224/train": {
      "expected": 5778,
      "actual": 5778
    },
    "ClsKLData/kneeKL224/val": {
      "expected": 826,
      "actual": 826
    },
    "ClsKLData/kneeKL224/test": {
      "expected": 1656,
      "actual": 1656
    },
    "ClsKLData/kneeKL224/auto_test": {
      "expected": 1526,
      "actual": 1526
    },
    "ClsKLData/kneeKL299/train": {
      "expected": 5778,
      "actual": 5778
    },
    "ClsKLData/kneeKL299/val": {
      "expected": 826,
      "actual": 826
    },
    "ClsKLData/kneeK

In [7]:
source_record = f'''KneeXrayData Mendeley Data release v1
DOI: 10.17632/56rmx5bjcr.1
Dataset page: {DATASET_PAGE}
License: CC BY 4.0
Archive filename: KneeXrayData.zip
Archive size: {EXPECTED_ARCHIVE_SIZE} bytes
Archive SHA-256: {EXPECTED_SHA256}

The release is organized from the Osteoarthritis Initiative and contains
KL classification crops plus full-image HDF5 knee-detection records.
'''
source_path = DRIVE_ROOT / 'SOURCE_AND_LICENSE.txt'
source_path.write_text(source_record, encoding='utf-8')

print('DATASET UPLOAD AND EXTRACTION COMPLETE')
print(f'Archive: {ARCHIVE_PATH}')
print(f'Dataset: {DATA_ROOT}')
print(f'Provenance: {source_path}')
print('You may disconnect the Colab runtime now.')


DATASET UPLOAD AND EXTRACTION COMPLETE
Archive: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/KneeXrayData.zip
Dataset: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData
Provenance: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/SOURCE_AND_LICENSE.txt
You may disconnect the Colab runtime now.
